# Week 1 Task: Data Acquisition, Cleaning, and Exploratory Analysis

**Name:** Hiya Saini  
**Enrollment No.:** A2305224345  

Dataset: Breast Cancer Wisconsin (Diagnostic), UCI Machine Learning Repository.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

## 2. Data Acquisition

The official source is the UCI Machine Learning Repository (dataset ID 17). For a reproducible local run, scikit-learn's bundled representation of the same WDBC dataset is loaded.

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df['diagnosis'] = df['target'].map({0:'malignant', 1:'benign'})
df.drop(columns=['target'], inplace=True)
print(df.shape)
df.head()

## 3. Initial Inspection

In [ ]:
print(df.info())
print('\nMissing values:\n', df.isna().sum().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nSummary statistics:')
display(df.describe())

## 4. Controlled Cleaning Demonstration

The source dataset is clean. Because the internship task explicitly asks us to demonstrate missing-value and duplicate handling, a small controlled amount of missingness and five duplicate rows is introduced into a working copy. The source dataset itself is never modified.

In [ ]:
working = df.copy()
rng = np.random.default_rng(42)
for col, n in [('mean radius',6), ('mean texture',6), ('mean area',5)]:
    idx = rng.choice(working.index, size=n, replace=False)
    working.loc[idx, col] = np.nan
working = pd.concat([working, working.iloc[rng.choice(working.index, 5)]], ignore_index=True)
print('Missing values before:', working.isna().sum().sum())
print('Duplicates before:', working.duplicated().sum())

In [ ]:
working = working.drop_duplicates().reset_index(drop=True)
for col in working.select_dtypes(include=np.number).columns:
    if working[col].isna().any():
        working[col] = working[col].fillna(working[col].median())
working['diagnosis'] = working['diagnosis'].astype('category')
print('Missing values after:', working.isna().sum().sum())
print('Duplicates after:', working.duplicated().sum())

## 5. Exploratory Analysis

In [ ]:
display(working.describe())
print(working['diagnosis'].value_counts())

### Visualization 1: Missing Values Before Cleaning

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
# The source has no missing values; the controlled working copy is used for the report.
working_missing = pd.Series({'mean radius':6, 'mean texture':6, 'mean area':5})
working_missing.plot(kind='bar', figsize=(8,5), title='Missing Values Before Cleaning')
plt.ylabel('Count'); plt.xticks(rotation=30); plt.show()

### Visualization 2: Class Distribution

In [ ]:
working['diagnosis'].value_counts().plot(kind='bar', figsize=(7,5), title='Diagnosis Class Distribution')
plt.ylabel('Number of Samples'); plt.show()

### Visualization 3: Histogram

In [ ]:
plt.figure(figsize=(8,5)); plt.hist(working['mean radius'], bins=25)
plt.title('Distribution of Mean Radius'); plt.xlabel('Mean Radius'); plt.ylabel('Frequency'); plt.show()

### Visualization 4: Box Plot

In [ ]:
data_box = [working.loc[working['diagnosis']=='benign','mean radius'],
            working.loc[working['diagnosis']=='malignant','mean radius']]
plt.figure(figsize=(8,5)); plt.boxplot(data_box, labels=['Benign','Malignant'])
plt.title('Mean Radius by Diagnosis'); plt.ylabel('Mean Radius'); plt.show()

### Visualization 5: Scatter Plot

In [ ]:
plt.figure(figsize=(8,5))
for label in ['benign','malignant']:
    sub = working[working['diagnosis']==label]
    plt.scatter(sub['mean radius'], sub['mean perimeter'], alpha=.55, label=label)
plt.xlabel('Mean Radius'); plt.ylabel('Mean Perimeter'); plt.title('Mean Radius vs Mean Perimeter')
plt.legend(); plt.show()

### Visualization 6: Correlation Heatmap

In [ ]:
features = ['mean radius','mean texture','mean perimeter','mean area','mean smoothness',
            'mean compactness','mean concavity','mean concave points','mean symmetry','mean fractal dimension']
corr = working[features].corr()
plt.figure(figsize=(10,8)); im=plt.imshow(corr.values, aspect='auto'); plt.colorbar(im)
plt.xticks(range(len(features)), features, rotation=70, ha='right', fontsize=8)
plt.yticks(range(len(features)), features, fontsize=8)
plt.title('Correlation Heatmap of Selected Numerical Features'); plt.tight_layout(); plt.show()

## 6. Key Insights

- The dataset contains 569 observations and 30 predictive features.
- The class distribution is moderately imbalanced toward benign cases.
- Mean radius shows a wider range for malignant cases and is strongly related to other size-related measurements.
- Mean radius and mean perimeter show a clear positive relationship.
- The correlation analysis indicates substantial multicollinearity among several geometric measurements.
- The controlled cleaning exercise demonstrates duplicate removal and median imputation without altering the original public source.